# 🔬 EDA & Exploration: Polyp Instance Segmentation

Self-contained experimentation environment for YOLO11-seg polyp detection.

**Goal**: Visually explore the dataset and conduct hyperparameter sweeping to find optimal training configurations.

In [ ]:
# === 1. Setup Environment ===
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics "ray[tune]" seaborn

import os
import glob
import matplotlib.pyplot as plt
from ultralytics import YOLO
import torch

DATASET_YAML = '/content/drive/MyDrive/data_lake/03_gold/016_polyp_fast_diag_dataset/dataset.yaml'
assert os.path.exists(DATASET_YAML), "Dataset not found!"


## 2. Visualize the Ground Truth Data
Explore the annotation mapping visually to ensure polygons match up.

In [ ]:
from IPython.display import Image, display
import random

val_images = glob.glob(DATASET_YAML.replace('dataset.yaml', 'val/images/*.*'))
if val_images:
    sample = random.choice(val_images)
    print(f"Sample Ground Truth Target: {sample}")
    display(Image(filename=sample, width=400))


## 3. Hyperparameter Sweep (Ray Tune)
Use YOLO's native `tune()` integration to sweep through Learning Rates, momentums, optimizer constraints, and augmentations.
**Tweak instructions:** Adjust `iterations` to set how many configurations to search, and `epochs` for the depth of each search.

In [ ]:
model = YOLO('yolo11n-seg.pt')

print("Starting tuning run...")
results = model.tune(
    data=DATASET_YAML,
    use_ray=True, 
    iterations=5,      # <-- TWEAK THIS: How many different hyperparam configurations to try
    epochs=15,          # <-- TWEAK THIS: Training epochs per iteration
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    project="/content/drive/MyDrive/models/tuning",
    name="016_polyp_tune"
)


## 4. Plotting Tuning Results
Extract scatter plots and evaluation charts from the Ray Tune CSV to evaluate performance.

In [ ]:
import pandas as pd
import seaborn as sns

results_path = '/content/drive/MyDrive/models/tuning/016_polyp_tune/tune_results.csv'

if os.path.exists(results_path):
    results_df = pd.read_csv(results_path)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    sns.scatterplot(data=results_df, x='lr0', y='metrics/mAP50-95(B)', ax=ax1, hue='momentum')
    ax1.set_title('Learning Rate vs BBox mAP')
    
    sns.scatterplot(data=results_df, x='weight_decay', y='metrics/mAP50-95(M)', ax=ax2, hue='lr0')
    ax2.set_title('Weight Decay vs Segmentation mAP')
    
    plt.show()
else:
    print(f"Results file not found at {results_path}. Did the tuning job finish?")
